In [0]:
# Databricks notebook source

import requests

from decimal import Decimal

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    to_date,
    date_format,
    trunc,
    current_timestamp,
    lit,
    max as spark_max
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType
)

In [0]:
# COMMAND ----------

dbutils.widgets.text(
    "load_type",
    "incremental_load"
)

load_type = (
    dbutils.widgets
    .get("load_type")
    .lower()
    .strip()
)

if load_type not in {
    "full_load",
    "incremental_load"
}:
    raise ValueError(
        "Invalid load_type. "
        "Use 'full_load' or 'incremental_load'."
    )

print(f"Load type: {load_type}")

In [0]:
# COMMAND ----------

API_URL = (
    "https://script.google.com/macros/s/"
    "AKfycbxYY0tYAuhKTOZlnhVhg9HFB3b4JiczEbrGtl5cgFAOSW63qOIWKKEJD1Onxttfz7PZ/"
    "exec"
)

BRONZE_TABLE = (
    "personal.finance.bronze_transactions"
)

response = requests.get(
    API_URL,
    timeout=120
)

response.raise_for_status()

api_response = response.json()

if api_response.get("status") != "success":
    raise Exception(
        api_response.get(
            "message",
            "Google Sheets API failed"
        )
    )

data = api_response.get("data", [])

print(f"Rows received from API: {len(data)}")

In [0]:
# COMMAND ----------

schema = StructType([
    StructField(
        "Date",
        StringType(),
        True
    ),
    StructField(
        "Amount",
        DecimalType(18, 2),
        True
    ),
    StructField(
        "Description",
        StringType(),
        True
    ),
    StructField(
        "Category",
        StringType(),
        True
    ),
    StructField(
        "Remark",
        StringType(),
        True
    ),
    StructField(
        "Year",
        StringType(),
        True
    ),
    StructField(
        "Month",
        StringType(),
        True
    ),
    StructField(
        "Transaction Type",
        StringType(),
        True
    )
])

In [0]:
# COMMAND ----------

for row in data:

    if row.get("Amount") is not None:
        row["Amount"] = Decimal(
            str(row["Amount"])
        )

df_source = spark.createDataFrame(
    data,
    schema=schema
)

In [0]:
# COMMAND ----------

df_source = (
    df_source
    .withColumnRenamed(
        "Date",
        "transaction_date"
    )
    .withColumnRenamed(
        "Amount",
        "amount"
    )
    .withColumnRenamed(
        "Description",
        "description"
    )
    .withColumnRenamed(
        "Category",
        "category"
    )
    .withColumnRenamed(
        "Remark",
        "remark"
    )
    .withColumnRenamed(
        "Year",
        "year"
    )
    .withColumnRenamed(
        "Month",
        "month"
    )
    .withColumnRenamed(
        "Transaction Type",
        "transaction_type"
    )
)

In [0]:
# COMMAND ----------

df_source = (
    df_source

    # Date
    .withColumn(
        "transaction_date",
        to_date(
            col("transaction_date"),
            "yyyy-MM-dd"
        )
    )

    # Amount
    .withColumn(
        "amount",
        col("amount").cast(
            "decimal(18,2)"
        )
    )

    # Text cleanup
    .withColumn(
        "description",
        trim(col("description"))
    )

    .withColumn(
        "category",
        trim(col("category"))
    )

    .withColumn(
        "remark",
        trim(col("remark"))
    )

    # Transaction type
    .withColumn(
        "transaction_type",
        upper(
            trim(
                col("transaction_type")
            )
        )
    )

    # Year
    .withColumn(
        "year",
        col("year").cast("integer")
    )

    # Derive month from transaction date
    .withColumn(
        "month",
        date_format(
            col("transaction_date"),
            "MMM-yyyy"
        )
    )

    # First day of month
    .withColumn(
        "month_date",
        trunc(
            col("transaction_date"),
            "month"
        )
    )

    # Audit columns
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )

    .withColumn(
        "source",
        lit("google_sheets")
    )
)

In [0]:
# COMMAND ----------

df_source.printSchema()

display(df_source.limit(20))

In [0]:
# COMMAND ----------

bronze_exists = spark.catalog.tableExists(
    BRONZE_TABLE
)

max_date = None

if bronze_exists:

    max_date = (
        spark.table(BRONZE_TABLE)
        .select(
            spark_max(
                "transaction_date"
            ).alias("max_date")
        )
        .first()["max_date"]
    )

print(
    f"Bronze max transaction date: {max_date}"
)

In [0]:
# COMMAND ----------

if load_type == "full_load":

    print("Running FULL LOAD")

    df_filtered = df_source

else:

    print("Running INCREMENTAL LOAD")

    if max_date is None:

        print(
            "No existing Bronze data found. "
            "Processing all source records."
        )

        df_filtered = df_source

    else:

        print(
            f"Processing records after "
            f"{max_date}"
        )

        df_filtered = df_source.filter(
            col("transaction_date") > max_date
        )

In [0]:
# COMMAND ----------

source_count = len(data)

rows_to_process = (
    df_filtered
    .limit(1)
    .count()
)

print(
    f"Rows received from API: {source_count}"
)

if bronze_exists:
    print(
        f"Bronze max date: {max_date}"
    )

if rows_to_process == 0:

    print(
        "No new records to process."
    )

else:

    process_count = df_filtered.count()

    print(
        f"Rows to process: {process_count}"
    )

In [0]:
# COMMAND ----------

if rows_to_process == 0:

    print(
        "Bronze table was not modified."
    )

elif load_type == "full_load":

    (
        df_filtered
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(BRONZE_TABLE)
    )

    print(
        "Bronze FULL LOAD completed."
    )

else:

    (
        df_filtered
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    print(
        "Bronze INCREMENTAL LOAD completed."
    )

In [0]:
# COMMAND ----------

display(
    spark.table(BRONZE_TABLE)
    .orderBy(
        col("transaction_date").desc()
    )
    .limit(20)
)